# Baseline Holt-Winters 2025H1 (Benchmark)

This notebook runs a Holt-Winters benchmark per zone on top-demand zones.

Notes:
- Data extraction from HDFS is distributed by Spark.
- Model fitting itself is done in pandas/statsmodels on driver as a benchmark.
- Outputs are written back to HDFS.

In [ ]:
import json
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

spark = (
    SparkSession.builder
    .appName("DemandPredictionHoltWintersBenchmark2025H1")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    print(f"\n===== CLUSTER UTILIZATION: {stage_name} =====")
    try:
        payload = json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = payload.get("workers", [])
        print("alive workers:", payload.get("aliveworkers"))
        print("active apps :", len(payload.get("activeapps", [])))
        for w in workers:
            print("worker", w.get("id"), "cores", f"{w.get('coresused', 0)}/{w.get('cores', 0)}", "memory", f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
    except Exception as e:
        print("Could not query Spark Master JSON:", e)

cluster_util("session_started")

In [ ]:
DENSE_PATH = "/user/data/feature_engineering/demand_prediction_2025H1_dense_10m"
OUT_BASE = "/user/data/results/demand_prediction/2025H1_holt_winters"
ZONE_COL = "PULocationID"
BIN_COL = "pickup_bin_10m"
TARGET_COL = "pickup_demand"
TOP_N_ZONES = 40
SEASONAL_PERIODS = 144

dense = spark.read.parquet(DENSE_PATH).cache()
print("Dense rows:", dense.count())
cluster_util("after_dense_read")

top_zones = [
    r[ZONE_COL]
    for r in dense.groupBy(ZONE_COL)
    .agg(F.sum(TARGET_COL).alias("total"))
    .orderBy(F.col("total").desc())
    .limit(TOP_N_ZONES)
    .collect()
]

print("Top zones selected:", len(top_zones))
pdf = (
    dense.where(F.col(ZONE_COL).isin(top_zones))
    .orderBy(ZONE_COL, BIN_COL)
    .toPandas()
)

pdf[BIN_COL] = pd.to_datetime(pdf[BIN_COL])
pdf[TARGET_COL] = pd.to_numeric(pdf[TARGET_COL], errors="coerce").fillna(0.0)
cluster_util("after_collect_top_zones")

In [ ]:
metrics = []
pred_rows = []

for zone_id, g in pdf.groupby(ZONE_COL):
    g = g.sort_values(BIN_COL).reset_index(drop=True)
    y = g[TARGET_COL].astype(float).to_numpy()

    if len(y) < (SEASONAL_PERIODS * 3):
        continue

    split_idx = int(len(y) * 0.7)
    split_idx = max(split_idx, SEASONAL_PERIODS * 2)
    split_idx = min(split_idx, len(y) - 1)

    y_train = y[:split_idx]
    y_test = y[split_idx:]
    ts_test = g[BIN_COL].iloc[split_idx:].to_numpy()

    try:
        hw = ExponentialSmoothing(
            y_train,
            trend="add",
            seasonal="add",
            seasonal_periods=SEASONAL_PERIODS,
            initialization_method="estimated",
        )
        fitted = hw.fit(optimized=True)
        pred = fitted.forecast(len(y_test))
    except Exception:
        continue

    pred = np.maximum(np.asarray(pred, dtype=float), 0.0)
    mae = float(np.mean(np.abs(y_test - pred)))
    rmse = float(np.sqrt(np.mean((y_test - pred) ** 2)))
    mask = y_test > 0
    mape = float(np.mean(np.abs((y_test[mask] - pred[mask]) / y_test[mask])) * 100.0) if mask.any() else None

    metrics.append({
        ZONE_COL: int(zone_id),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
    })

    pred_rows.extend([
        {
            ZONE_COL: int(zone_id),
            BIN_COL: ts_test[i],
            "actual": float(y_test[i]),
            "prediction": float(pred[i]),
        }
        for i in range(len(y_test))
    ])

metrics_pdf = pd.DataFrame(metrics)
pred_pdf = pd.DataFrame(pred_rows)

if len(metrics_pdf) == 0:
    raise ValueError("No zone produced a valid Holt-Winters fit.")

display(metrics_pdf.sort_values("MAPE"))
print("Avg MAPE across fitted zones:", metrics_pdf["MAPE"].dropna().mean())

In [ ]:
metrics_sdf = spark.createDataFrame(metrics_pdf)
pred_sdf = spark.createDataFrame(pred_pdf)

metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/predictions")

print("Saved outputs:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")
cluster_util("after_hw_write")

In [ ]:
best_zone = int(metrics_pdf.sort_values("MAPE").iloc[0][ZONE_COL])
tmp = pred_pdf[pred_pdf[ZONE_COL] == best_zone].copy().sort_values(BIN_COL)

plt.figure(figsize=(16, 5))
plt.plot(tmp[BIN_COL], tmp["actual"], label="actual", linewidth=1.0)
plt.plot(tmp[BIN_COL], tmp["prediction"], label="holt_winters", linewidth=1.0)
plt.title(f"Holt-Winters benchmark - Zone {best_zone}")
plt.xlabel("time")
plt.ylabel("pickups / 10m")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()